 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

### آموزش مفهومی و عملی با Python

در این Notebook یاد می‌گیریم متن چگونه به بردار عددی تبدیل می‌شود، چطور شباهت بردارها را اندازه بگیریم و یک نمونه ساده از **Semantic Search** بسازیم.

**هدف:** درک نقش Embedding در RAG و پیاده‌سازی یک نمونه کوچک با Python.


## فهرست مطالب
1. Embedding چیست؟
2. Vector و Vector Space
3. نمایش معنایی متن
4. Embedding Models
5. Document Embeddings
6. Query Embedding
7. Cosine Similarity و Dot Product
8. Embedding Dimensions
9. انتخاب مدل
10. پیاده‌سازی Semantic Search
11. نقش Embedding در معماری RAG
12. جمع‌بندی و تمرین

---

## ۱. Embedding چیست؟

فرض کنید در سند منابع انسانی نوشته شده است:

**«کارکنان شرکت سالانه ۲۶ روز مرخصی استحقاقی دارند.»**

کاربر می‌پرسد:

**«من در سال چند روز می‌توانم مرخصی بگیرم؟»**

این دو متن از نظر مفهوم مرتبط‌اند، اما کلمات یکسانی ندارند. جست‌وجوی صرفاً کلمه‌محور (Keyword Search) ممکن است ارتباط آن‌ها را به‌خوبی پیدا نکند.

**Embedding** روشی برای تبدیل داده‌هایی مانند متن به نمایش عددی است. خروجی معمولاً آرایه‌ای از اعداد به نام **Vector** است.

نمونه فرضی یک بردار:
</div>

In [1]:
example_vector = [0.21, -0.45, 0.83, 0.12, -0.67]
print(example_vector)
print("Dimensions:", len(example_vector))

[0.21, -0.45, 0.83, 0.12, -0.67]
Dimensions: 5


 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

اعداد بالا فقط نمونه آموزشی هستند و از مدل واقعی تولید نشده‌اند. Embedding امکان مقایسه ریاضی نمایش عددی متن‌ها را فراهم می‌کند.

**نکته:** Embedding Model پاسخ طبیعی تولید نمی‌کند؛ وظیفه آن تولید نمایش برداری برای مقایسه و جست‌وجو است.

## ۲. Vector و Vector Space

**Vector** مجموعه‌ای مرتب از اعداد است. تعداد اعداد آن را **Dimension** یا بُعد می‌نامیم. بردار زیر سه بُعد دارد:
</div>

In [2]:
vector = [0.2, 0.5, -0.1]
print("Vector:", vector)
print("Dimensions:", len(vector))

Vector: [0.2, 0.5, -0.1]
Dimensions: 3


 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در مدل‌های واقعی، تعداد ابعاد می‌تواند چند صد یا چند هزار عدد باشد.

**Vector Space** فضای ریاضی‌ای است که بردارها در آن قرار می‌گیرند. می‌توان هر متن را نقطه‌ای در این فضا تصور کرد. مدل مناسب تلاش می‌کند متن‌های دارای معنای مشابه را نزدیک‌تر نمایش دهد؛ اما نزدیکی بردارها تضمین نمی‌کند که دو متن دقیقاً هم‌معنی باشند.

## ۳. Semantic Representation of Text

نمایش معنایی یعنی مدل تلاش می‌کند مفهوم و زمینه متن را در بردار آن منعکس کند.

مثلاً:
- «ماشین من خراب شده است.»
- «خودروی من مشکل فنی دارد.»

این دو عبارت از نظر واژگانی متفاوت‌اند، اما از نظر معنایی به هم نزدیک‌اند. در مقابل، «امروز هوا بارانی است» موضوع متفاوتی دارد.

Embedding صرفاً فرهنگ لغت مترادف‌ها نیست؛ مدل با الگوهایی که در داده‌های آموزشی آموخته، متن را نمایش می‌دهد و زمینه (Context) نیز اهمیت دارد.

هر بُعد بردار لزوماً یک مفهوم مشخص و قابل تفسیر برای انسان نیست.

## ۴. Embedding Models

**Embedding Model** مدلی است که متن را به Vector تبدیل می‌کند. مدل‌های مختلف ممکن است برای یک متن بردارهایی با مقادیر و ابعاد متفاوت بسازند.

در RAG دو کاربرد اصلی داریم:

1. **Document Embedding:** تبدیل Chunkهای اسناد به بردار.
2. **Query Embedding:** تبدیل سؤال کاربر به بردار.

برای مقایسه معتبر، باید مدل و تنظیمات دو سمت سازگار باشند. معمولاً از همان مدل برای اسناد و سؤال‌ها استفاده می‌کنیم.

---

## ۵. تولید Document Embeddings

فرض می‌کنیم اسناد را قبلاً به Chunk تقسیم کرده‌ایم. هر Chunk را به مدل می‌دهیم و بردار حاصل را همراه متن و Metadata نگه می‌داریم.

در این مثال از کتابخانه `sentence-transformers` و مدل چندزبانه `paraphrase-multilingual-MiniLM-L12-v2` استفاده می‌کنیم. مدل برای جمله‌های چندزبانه طراحی شده و در این مثال از متن فارسی استفاده می‌کنیم.

### نصب کتابخانه‌ها
</div>

In [3]:
%pip install -q sentence-transformers scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.


 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

بارگذاری اولیه مدل ممکن است به اینترنت و دانلود فایل‌های مدل نیاز داشته باشد.

### بارگذاری مدل و تولید بردار اسناد
</div>

In [12]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)

documents = [
    "کارکنان شرکت سالانه ۲۶ روز مرخصی استحقاقی دارند.",
    "هزینه بیمه تکمیلی کارکنان طبق قرارداد شرکت پرداخت می‌شود.",
    "حقوق کارکنان در پایان هر ماه واریز می‌شود.",
    "برای درخواست مرخصی باید فرم مربوطه ثبت شود."
]

document_embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings shape:", document_embeddings.shape)
print("First vector, first 8 values:", document_embeddings[0][:8])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2227.81it/s]


Embeddings shape: (4, 384)
First vector, first 8 values: [ 0.00031212 -0.03598768 -0.06343395 -0.05437794  0.07421836  0.00780955
  0.04671458 -0.02848574]


 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در خروجی `shape`:
- عدد اول تعداد متن‌ها یا Chunkهاست.
- عدد دوم تعداد ابعاد هر بردار است.

مثلاً شکل `(4, 384)` یعنی چهار بردار داریم و هر بردار ۳۸۴ عدد دارد؛ مقدار واقعی را از خروجی مدل می‌خوانیم و نباید حدس بزنیم.

گزینه `normalize_embeddings=True` بردارها را نرمال می‌کند؛ یعنی طول هر بردار را به ۱ می‌رساند. این کار برای برخی مقایسه‌های برداری مفید است.

## ۶. تولید Query Embedding

هنگام دریافت سؤال کاربر، آن را با همان مدل به Vector تبدیل می‌کنیم.
</div>

In [5]:
query = "من در هر سال چند روز مرخصی می‌توانم بگیرم؟"

query_embedding = model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Query shape:", query_embedding.shape)
print("First 8 values:", query_embedding[:8])

Query shape: (384,)
First 8 values: [ 0.01273342 -0.02834286  0.02306866 -0.0390529   0.01230902  0.05065471
 -0.01767832  0.00282753]


 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

اکنون یک بردار برای سؤال و تعدادی بردار برای اسناد داریم. باید میزان شباهت آن‌ها را محاسبه کنیم.

## ۷. معیارهای شباهت (Similarity Metrics)

### ۷.۱. Cosine Similarity

این معیار زاویه بین دو بردار را بررسی می‌کند. مقدار آن در حالت کلی بین `-1` و `1` است:
- نزدیک `1`: جهت دو بردار مشابه است.
- نزدیک `0`: شباهت زاویه‌ای کمی دارند.
- نزدیک `-1`: جهت آن‌ها تقریباً مخالف است.

فرمول:

\[
cosine(A,B)=\frac{A \cdot B}{\|A\|\|B\|}
\]

### ۷.۲. Dot Product

حاصل‌ضرب داخلی دو بردار:

\[
A \cdot B = \sum_{i=1}^{n} A_i B_i
\]

مقدار Dot Product به طول بردارها و زاویه بین آن‌ها وابسته است.

**نکته:** اگر هر دو بردار نرمال شده باشند و طولشان ۱ باشد، Dot Product با Cosine Similarity برابر می‌شود.

### محاسبه شباهت با scikit-learn
</div>

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(
    query_embedding.reshape(1, -1),
    document_embeddings
)[0]

for document, score in zip(documents, scores):
    print(f"Score: {score:.4f} | {document}")

Score: 0.6180 | کارکنان شرکت سالانه ۲۶ روز مرخصی استحقاقی دارند.
Score: 0.1012 | هزینه بیمه تکمیلی کارکنان طبق قرارداد شرکت پرداخت می‌شود.
Score: 0.4042 | حقوق کارکنان در پایان هر ماه واریز می‌شود.
Score: 0.3624 | برای درخواست مرخصی باید فرم مربوطه ثبت شود.


 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

امتیاز بالاتر در این مقایسه یعنی بردار سند از نظر این معیار به بردار سؤال نزدیک‌تر است. این امتیاز **احتمال درست بودن پاسخ نیست** و نباید آن را درصد اطمینان تلقی کرد.

### مرتب‌سازی بر اساس شباهت
</div>

In [7]:
ranked_results = sorted(
    zip(documents, scores),
    key=lambda item: item[1],
    reverse=True
)

for rank, (document, score) in enumerate(ranked_results, start=1):
    print(f"{rank}. Score={score:.4f}")
    print(document)
    print()

1. Score=0.6180
کارکنان شرکت سالانه ۲۶ روز مرخصی استحقاقی دارند.

2. Score=0.4042
حقوق کارکنان در پایان هر ماه واریز می‌شود.

3. Score=0.3624
برای درخواست مرخصی باید فرم مربوطه ثبت شود.

4. Score=0.1012
هزینه بیمه تکمیلی کارکنان طبق قرارداد شرکت پرداخت می‌شود.



 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

## ۸. Embedding Dimensions

**Embedding Dimension** تعداد اعداد موجود در هر بردار است. ابعاد بیشتر الزاماً به معنی کیفیت بهتر نیست.

ابعاد روی فضای ذخیره‌سازی و هزینه جست‌وجو اثر دارند؛ اما انتخاب مدل باید با آزمایش کیفیت Retrieval روی داده‌های واقعی انجام شود.

</div>

In [8]:
print("Number of documents:", document_embeddings.shape[0])
print("Embedding dimensions:", document_embeddings.shape[1])

Number of documents: 4
Embedding dimensions: 384


 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

## ۹. انتخاب Embedding Model

برای انتخاب مدل مناسب فقط به تعداد ابعاد یا محبوبیت آن توجه نکنید:

| معیار | دلیل اهمیت |
|---|---|
| پشتیبانی زبان | کیفیت مدل برای فارسی و زبان اسناد |
| کیفیت Retrieval | رتبه‌بندی درست Chunkهای مرتبط |
| ابعاد بردار | اثر بر ذخیره‌سازی و هزینه |
| سرعت | زمان پردازش اسناد و جست‌وجو |
| هزینه | هزینه استفاده از مدل |
| محدودیت ورودی | حداکثر طول متن قابل پردازش |
| مجوز | سازگاری شرایط استفاده با پروژه |

بهترین روش، ارزیابی چند مدل با مجموعه‌ای از سؤال‌ها و اسناد واقعی پروژه است.

---

## ۱۰. پیاده‌سازی Semantic Search

تابع زیر سؤال را Embedding می‌کند، شباهت آن را با بردار اسناد می‌سنجد و `top_k` نتیجه برتر را برمی‌گرداند.

این نمونه یک جست‌وجوی معنایی کوچک است؛ هنوز RAG کامل نیست، چون به LLM وصل نشده و پاسخ تولید نمی‌کند.
</div>

In [9]:
def semantic_search(query, documents, model, top_k=2):
    query_vector = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    document_vectors = model.encode(
        documents,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_vector.reshape(1, -1),
        document_vectors
    )[0]

    ranked_indices = scores.argsort()[::-1][:top_k]

    return [
        {
            "text": documents[index],
            "score": float(scores[index])
        }
        for index in ranked_indices
    ]


question = "قوانین مربوط به مرخصی کارکنان چیست؟"

results = semantic_search(
    question,
    documents,
    model,
    top_k=2
)

for result in results:
    print(f"Score: {result['score']:.4f}")
    print(result["text"])
    print("-" * 40)

Score: 0.6218
کارکنان شرکت سالانه ۲۶ روز مرخصی استحقاقی دارند.
----------------------------------------
Score: 0.5994
برای درخواست مرخصی باید فرم مربوطه ثبت شود.
----------------------------------------


 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

## ۱۱. نقش Embedding در معماری RAG

فرآیند کلی به این شکل است:

1. **Ingestion:** دریافت اسناد.
2. **Chunking:** تقسیم اسناد به بخش‌های کوچک‌تر.
3. **Document Embedding:** تبدیل هر Chunk به Vector.
4. **Storage:** ذخیره Vector همراه متن و Metadata در Vector Database.
5. **Query Embedding:** تبدیل سؤال کاربر به Vector.
6. **Vector Search:** پیدا کردن بردارهای نزدیک و بازیابی Chunkها.
7. **Generation:** ارسال سؤال و متن‌های بازیابی‌شده به LLM برای تولید پاسخ.

### چه اطلاعاتی در Vector Database ذخیره می‌شود؟

معمولاً برای هر Chunk این اطلاعات را نگه می‌داریم:
- متن Chunk
- بردار Embedding
- Metadata مانند شناسه سند، نام فایل یا شماره صفحه

پس از پیدا کردن بردار مناسب، متن و منبع مربوط به آن نیز قابل بازیابی است.

**نکته مهندسی:** اگر مدل Embedding را تغییر دهیم، بردارهای جدید لزوماً با بردارهای قبلی قابل مقایسه نیستند. معمولاً باید اسناد را با مدل جدید دوباره Embedding کنیم و نسخه مدل را در فرآیند نگهداری داده‌ها مشخص کنیم.

## ۱۲. محدودیت‌های Embedding

- ممکن است متن مرتبط را پیدا نکند یا نتیجه نامرتبط برگرداند.
- شناسه‌ها، کدها و عبارت‌های دقیق گاهی با Keyword Search بهتر پیدا می‌شوند.
- کیفیت به مدل، زبان، Chunking و داده‌ها وابسته است.
- شباهت معنایی به معنی صحت اطلاعات یا پاسخ نیست.
- مجوز دسترسی باید جداگانه بررسی شود؛ پیدا شدن یک سند به معنی مجاز بودن کاربر برای مشاهده آن نیست.

---

## ۱۳. جمع‌بندی

| اصطلاح | مفهوم |
|---|---|
| Embedding | نمایش عددی داده |
| Vector | مجموعه مرتب اعداد |
| Vector Space | فضای ریاضی بردارها |
| Embedding Model | مدل تولیدکننده بردار |
| Document Embedding | بردار مربوط به سند یا Chunk |
| Query Embedding | بردار مربوط به سؤال |
| Cosine Similarity | معیار مقایسه جهت دو بردار |
| Dot Product | حاصل‌ضرب داخلی |
| Dimensions | تعداد اعداد هر بردار |
| Semantic Search | جست‌وجو بر اساس شباهت معنایی |

## ۱۴. تمرین عملی

با استفاده از همین Notebook:

1. سه سند جدید درباره بیمه تکمیلی، حقوق و مرخصی به `documents` اضافه کن.
2. یک سؤال درباره بیمه تکمیلی بنویس.
3. تابع `semantic_search` را با `top_k=3` اجرا کن.
4. بررسی کن سند مرتبط در رتبه اول قرار گرفته است یا نه.
5. سؤال را با عبارت‌بندی متفاوت دوباره امتحان کن و نتایج را مقایسه کن.

**معیار موفقیت:** بتوانی توضیح بدهی چرا یک سند رتبه بالاتری گرفته و چه عواملی ممکن است باعث بازیابی نامناسب شوند.

## منابع و مستندات

- [Sentence Transformers — Quickstart](https://www.sbert.net/docs/quickstart.html)
- [Sentence Transformers — Semantic Search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)
- [scikit-learn — cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)
- [Hugging Face — Model Card](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)

</div>